# Train A 125M GPT-Style Model On FineWeb-Edu

<a href="https://colab.research.google.com/github/openlanguagemodel/openlanguagemodel/blob/main/notebooks/02_train_125m_fineweb_edu_colab.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

This notebook builds a GPT-2-small-size model, streams FineWeb-Edu
from Hugging Face, and runs a budget-aware training smoke run.

The default step count is intentionally small. Treat it as a
preflight check, then increase the steps on rented GPU hardware.

## Install OLM

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("olm") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/openlanguagemodel/openlanguagemodel.git",
    ])

## Imports

In [ ]:
import math
import random
import time

import torch

from olm.data.datasets import DataLoader, FineWebEduDataset
from olm.data.tokenization import HFTokenizer
from olm.models.openai import GPT2Model
from olm.train import AutoTrainer
from olm.train.optim import AdamW

seed = 42
random.seed(seed)
torch.manual_seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("For real 125M training, switch Colab to a GPU runtime.")

## Budget Knobs

Keep `MAX_STEPS` low for a smoke run. For a real run, increase it
after the first batches are stable. Cost depends on the GPU provider,
so the useful number to track here is tokens processed.

In [ ]:
CONTEXT_LENGTH = 1024
MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 32
MAX_STEPS = 20

effective_batch = MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS
tokens_per_step = effective_batch * CONTEXT_LENGTH
planned_tokens = tokens_per_step * MAX_STEPS

print("effective batch:", effective_batch)
print("tokens per optimizer step:", f"{tokens_per_step:,}")
print("planned tokens in this notebook run:", f"{planned_tokens:,}")

## Tokenizer And FineWeb-Edu Stream

In [ ]:
tokenizer = HFTokenizer("gpt2")

dataset = FineWebEduDataset(
    tokenizer=tokenizer,
    subset="sample-10BT",
    split="train",
    context_length=CONTEXT_LENGTH,
    streaming=True,
    shuffle=True,
    seed=seed,
)

loader = DataLoader(
    dataset,
    batch_size=MICRO_BATCH_SIZE,
    num_workers=0,
    pin_memory=device.startswith("cuda"),
)

x, y = next(iter(loader))
print("batch:", tuple(x.shape), tuple(y.shape))
print(tokenizer.decode(x[0][:200]))

## Build The 125M GPT-Style Model

In [ ]:
model = GPT2Model(
    vocab_size=tokenizer.vocab_size,
    embed_dim=768,
    num_layers=12,
    num_heads=12,
    max_seq_len=CONTEXT_LENGTH,
    dropout=0.1,
)

params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters: {params:,}")
print(f"trainable: {trainable:,}")
print("approx size fp32:", f"{params * 4 / 1e9:.2f} GB")

## Train A Short Preflight Run

In [ ]:
trainer = None
losses = []

if device == "cpu":
    print("Skipping 125M training on CPU. In Colab, choose Runtime -> Change runtime type -> GPU.")
else:
    trainer = AutoTrainer(
        model,
        AdamW,
        loader,
        device="auto",
        context_length=CONTEXT_LENGTH,
        learning_rate=3e-4,
        weight_decay=0.1,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        use_amp=True,
        grad_clip_norm=1.0,
        preset="balanced",
        verbose=True,
    )

    start = time.time()
    losses = trainer.train(epochs=1, max_steps=MAX_STEPS, log_interval=5)
    elapsed = time.time() - start

    print("final loss:", losses[-1])
    print("elapsed minutes:", elapsed / 60)
    print("tokens processed:", f"{trainer.total_tokens_processed:,}")

## Scale The Run

After the preflight works:

- Increase `MAX_STEPS`.
- Keep `MICRO_BATCH_SIZE` small if memory is tight.
- Increase `GRAD_ACCUM_STEPS` to raise the effective batch size.
- Use `AutoTrainer(..., preset="memory_efficient")` if the model is
  close to GPU memory limits.
- Save checkpoints periodically for longer rented-GPU runs.

In [ ]:
if trainer is None:
    print("No checkpoint saved because training was skipped.")
else:
    checkpoint = {
        "model_state_dict": trainer.model.state_dict(),
        "optimizer_state_dict": trainer.optimizer.state_dict(),
        "losses": losses,
        "context_length": CONTEXT_LENGTH,
        "max_steps": MAX_STEPS,
    }
    torch.save(checkpoint, "gpt2_125m_fineweb_edu_preflight.pt")
    print("saved gpt2_125m_fineweb_edu_preflight.pt")